# # rama
#
# > `/learn` in ramabana: the same tutor, in the terminal, beside your code.

In [ ]:
#| default_exp rama

In [ ]:
from fastcore.test import *

In [ ]:
#| export
import os, time
from fastcore.utils import *
from abhyasa.store import Learning
from abhyasa.model import DEPTHS
from abhyasa.tutor import Tutor, Session

# A ramabana extension is a file in `~/.config/ramabana/extensions/` with a `setup(reg)`. To turn this one on, make
# `~/.config/ramabana/extensions/abhyasa.py` with one line, `from abhyasa.rama import setup`, and install abhyasa where
# ramabana runs.
#
# | command | what it does |
# |---|---|
# | `/learn [topic or slug]` | start a session, on one concept when you name it |
# | `/a <answer>` | answer; put think-aloud after a line with `--` |
# | `/hint` | the next hint |
# | `/idk` | "I don't know" |
# | `/teach` | give up on this question and be taught |
# | `/due` | what is due |
#
# The model is ramabana's `agent.oneshot`, so `[jobs]` in `config.toml` routes it. Two tools let the coding agent add to
# the graph while you code: `learning_note` files a concept you just met, and `learning_due` lists what is due.

In [ ]:
#| export
_st = AttrDict(lr=None, tutor=None, sess=None)

def _learning():
    if _st.lr is None: _st.lr = Learning(os.getenv('ABHYASA_VAULT') or os.getenv('VISHALAKSHI_VAULT'))
    return _st.lr

def _tutor(agent):
    if _st.tutor is None: _st.tutor = Tutor(_learning(), lambda p, sp='': agent.oneshot(p, sp, job='oneshot'))
    return _st.tutor

def show_q(q) -> str:
    "A question as terminal text."
    if q is None: return 'Nothing to learn or review right now. `/learn <topic>` names one.'
    opts = ''.join(f'\n  {i}. {o}' for i, o in enumerate(q.options)) if q.get('options') else ''
    return f"**{q.name}** · {DEPTHS[q.depth]} · _{q.why}_\n\n{q.question}{opts}\n\n`/a <answer>` · `/hint` · `/idk`"

def show_r(r) -> str:
    "A result as terminal text."
    if not r.done: return f"Not yet. {r.feedback}\n\n{r.tries_left} more tries, or `/teach`."
    out = [f"**{r.outcome}**", r.feedback, r.lesson, f"Floor now {DEPTHS[r.floor]}. `/learn` for the next one."]
    return '\n\n'.join(o for o in out if o)

In [ ]:
#| export
def find(lr, arg) -> str|None:
    "The slug for `arg`: a slug, else the best concept hit in search, else None."
    if not arg: return None
    if lr.concept(arg): return arg
    hs = lr.search(arg, kind='concept', limit=1)
    if not hs: return None
    r = first(lr.v.t.docs(where='id=?', where_args=[hs[0]['doc_id']]))
    return r['source'].rsplit('/', 1)[-1] if r else None

def learn(agent, arg=''):
    t = _tutor(agent)
    focus = find(t.learning, arg.strip())
    if arg.strip() and not focus: return f'No concept matches {arg!r}. Add a source in the abhyasa app, or let the agent `learning_note` one.'
    _st.sess = Session(t, focus=focus)
    return show_q(_st.sess.next())

def _need():
    if not (_st.sess and _st.sess.q): raise ValueError('No open question. Start with `/learn`.')
    return _st.sess

def answer(agent, arg=''):
    ans, _, think = arg.partition('\n--')
    return show_r(_need().answer(ans.strip(), think.strip(), 70))

def hint(agent, arg=''): return _need().hint()
def idk(agent, arg=''): return show_r(_need().answer(dont_know=True))
def teach(agent, arg=''): return show_r(_need().give_up())

def due(agent=None, arg=''):
    cs = [c for c in _learning().concepts() if c.seen and c.due and c.due <= time.time()]
    return '\n'.join(f'- {c.name} ({c.subject})' for c in cs) or 'Nothing is due.'

In [ ]:
#| export
def learning_note(name:str, statement:str, subject:str='code', prereqs:str=''):
    "File a concept the user met while coding, so abhyasa can ask about it later. `prereqs` is comma-separated slugs."
    c = _learning().put(name, statement, subject, prereqs=[p.strip() for p in prereqs.split(',') if p.strip()],
                        provenance=[dict(src='ramabana')])
    return f'Filed {c.name} ({c.slug}).'

def learning_due() -> str:
    "The concepts the user has due for review."
    return due()

def _safe(fn, agent, arg):
    try: return fn(agent, arg)
    except ValueError as e: return str(e)

def setup(reg):
    "Register abhyasa's commands and tools with ramabana."
    for nm, fn, h in (('learn', learn, 'start a learning session'), ('a', answer, 'answer the open question'),
                      ('hint', hint, 'the next hint'), ('idk', idk, "say you don't know"),
                      ('teach', teach, 'give up and be taught'), ('due', due, 'what is due for review')):
        reg.command(nm, lambda agent, arg='', fn=fn: _safe(fn, agent, arg), help=h)
    reg.tool(learning_note); reg.tool(learning_due)

# ## Tests
#
# A stand-in registry and agent: commands go in a dict, and `oneshot` is the fake model.

In [ ]:
from abhyasa.store import temp_learning
from abhyasa.llm import Fake
fake = Fake({'You grade one answer': dict(correct=True, grade=9, first_error='', misconception='', feedback='Yes.')},
            dflt=dict(question='Why sort by score?', answer='To rank.', rubric=['rank']))
class Reg:
    def __init__(self): self.commands, self.tools = {}, []
    def command(self, n, f, help=''): self.commands[n] = f
    def tool(self, f): self.tools.append(f)
agent = AttrDict(oneshot=lambda p, sp='', job='': fake(p, sp))
_st.update(lr=temp_learning(), tutor=None, sess=None)
reg = Reg(); setup(reg)
test_eq(sorted(reg.commands), ['a', 'due', 'hint', 'idk', 'learn', 'teach'])
test_eq(reg.commands['a'](agent, 'x'), 'No open question. Start with `/learn`.')
assert 'Filed Ranking' in learning_note('Ranking', 'Order items by a score.', 'ml')
assert 'Why sort by score?' in reg.commands['learn'](agent, 'ranking')
assert '**solid**' in reg.commands['a'](agent, 'to rank them\n-- I think order matters')
test_eq(learning_due(), 'Nothing is due.')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()